# CURE-OR++ Open-Weight VLM Real-Transfer GPU Pilot

This notebook runs an open-weight VLM over the CURE-OR++ real-transfer v0.2
prompt pack. It is intended to produce a reproducible VLM baseline without
paid frontier API calls.

Default model: `HuggingFaceTB/SmolVLM2-500M-Video-Instruct`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd

os.environ.setdefault('HF_HOME', '/tmp/cure_or_pp_hf_home')
PROMPT_PACK_RELATIVE = Path('reports/vlm_api_track_v01_prompt_pack.jsonl')
DATASET_CANDIDATES = [
    Path('/kaggle/input/cure-or-pp-vlm-real-transfer-v02-private'),
    Path('/kaggle/input/cure-or-plus-plus-vlm-real-transfer-v02-private'),
    Path('/kaggle/input/datasets/yaroslavkholmirzayev/cure-or-pp-vlm-real-transfer-v02-private'),
]

def list_input_roots():
    input_root = Path('/kaggle/input')
    if not input_root.exists():
        return []
    return sorted(path for path in input_root.iterdir() if path.is_dir())

def find_data_root():
    input_root = Path('/kaggle/input')
    for path in DATASET_CANDIDATES:
        if (path / PROMPT_PACK_RELATIVE).exists():
            return path
    if input_root.exists():
        prompt_pack_candidates = sorted(input_root.rglob(str(PROMPT_PACK_RELATIVE)))
        print('Prompt pack candidates:', [str(path) for path in prompt_pack_candidates[:20]])
        if prompt_pack_candidates:
            return prompt_pack_candidates[0].parent.parent
    return None

print('Python:', sys.version)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not available')
print('Kaggle input roots:', [path.name for path in list_input_roots()])

DATA_ROOT = find_data_root()
if DATA_ROOT is None:
    raise FileNotFoundError(
        'Attach the private CURE-OR++ VLM real-transfer dataset to this notebook. '
        f'Checked explicit candidates: {[str(path) for path in DATASET_CANDIDATES]}; '
        f'available /kaggle/input roots: {[path.name for path in list_input_roots()]}'
    )

print('DATA_ROOT:', DATA_ROOT)
print('Prompt pack:', DATA_ROOT / 'reports/vlm_api_track_v01_prompt_pack.jsonl')


In [ ]:
%pip install -q --no-cache-dir --force-reinstall --index-url https://download.pytorch.org/whl/cu121 "torch==2.5.1+cu121" "torchvision==0.20.1+cu121"
%pip install -q --no-cache-dir -U "transformers>=4.57,<5" "accelerate>=1.8,<2" "pillow<12" num2words


In [ ]:
import transformers
import torch

PROMPT_PACK = DATA_ROOT / 'reports/vlm_api_track_v01_prompt_pack.jsonl'
RUNNER = DATA_ROOT / 'scripts/run_hf_vlm.py'
EVALUATOR = DATA_ROOT / 'scripts/evaluate_vlm_response_pack.py'

for required_path in [PROMPT_PACK, RUNNER, EVALUATOR]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

print('transformers:', transformers.__version__)
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('torch cuda arch list:', torch.cuda.get_arch_list())
    print('torch device capability:', torch.cuda.get_device_capability())

MODEL_ID = 'HuggingFaceTB/SmolVLM2-500M-Video-Instruct'
RUN_LIMIT = 210
RESPONSES = Path('/kaggle/working/vlm_responses_smolvlm2_500m.jsonl')

cmd = [
    sys.executable,
    str(RUNNER),
    '--prompt-pack', str(PROMPT_PACK),
    '--model', MODEL_ID,
    '--output', str(RESPONSES),
    '--cache-dir', '/kaggle/working/vlm_response_cache',
    '--limit', str(RUN_LIMIT),
    '--force',
    '--max-tokens', '8',
    '--dtype', 'float16',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(DATA_ROOT))


In [ ]:
MODEL_SUMMARY = Path('/kaggle/working/vlm_smolvlm2_model_summary.csv')
RECIPE_TABLE = Path('/kaggle/working/vlm_smolvlm2_recipe_table.csv')
LABEL_TABLE = Path('/kaggle/working/vlm_smolvlm2_label_table.csv')
AUDIT_TABLE = Path('/kaggle/working/vlm_smolvlm2_audit.csv')

cmd = [
    sys.executable,
    str(EVALUATOR),
    '--prompt-pack', str(PROMPT_PACK),
    '--responses', str(RESPONSES),
    '--model-summary', str(MODEL_SUMMARY),
    '--recipe-table', str(RECIPE_TABLE),
    '--label-table', str(LABEL_TABLE),
    '--audit-table', str(AUDIT_TABLE),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(DATA_ROOT))


In [ ]:
pd.read_csv(MODEL_SUMMARY)


In [ ]:
pd.read_csv(RECIPE_TABLE)


In [ ]:
pd.read_csv(LABEL_TABLE).sort_values(['real_accuracy', 'label']).head(20)
